# ScholarAI Colab Training Notebook
## Academic score prediction for the Flask `predicted_score` column

This notebook is designed for your **ScholarAI Flask app**.

### Goal
Train a proper **regression model** that predicts an academic score from website-compatible inputs, then export:

- `scholarai_academic_score_pipeline.joblib`
- `scholarai_model_metadata.json`
- `scholarai_business_rules.json`

These exported files are what your Flask website will use for **live prediction** in the **Admin → Predictions → Predicted Score** column.

### Website alignment
Your current form collects:

- `term1_score`
- `term2_score`
- `term3_score`
- `attendance_rate`

So this notebook forces the final model to use the same website-compatible feature set:

- `attendance_rate`
- `term1_score`
- `term2_score`
- `term3_score`
- `terminal_avg`

### Important design choice
The ML model predicts **academic score only**.
Your Flask app should continue using its **business risk logic** (risk level, dues, complaints, trend, recommendations) after the score is predicted.

## 1. Install dependencies

This notebook uses `kagglehub` to download a public Kaggle dataset and `scikit-learn` for training.

If Kaggle authentication is needed, the recommended approaches are:

- `kagglehub.login()`
- `KAGGLE_API_TOKEN` environment variable / Colab secret
- legacy `kaggle.json`

The notebook tries public datasets in order and picks the first compatible one.

In [ ]:

# If running in Google Colab, uncomment the next line if needed:
# !pip -q install -U kagglehub kagglehub[pandas-datasets] scikit-learn joblib openpyxl

In [ ]:

import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 2. Kaggle authentication and dataset download

The notebook tries a shortlist of public datasets relevant to your specification:
- attendance
- exam/internal scores
- assignment/performance signals
- final score / semester performance

It then automatically picks the most compatible dataset file.

In [ ]:

try:
    import kagglehub
    print("kagglehub imported successfully.")
except Exception as e:
    raise RuntimeError("Install kagglehub first using the pip cell above.") from e

# Optional interactive login if public download still asks for consent.
# kagglehub.login()

In [ ]:

PREFERRED_DATASETS = [
    {
        "handle": "sonalshinde123/student-academic-performance-dataset",
        "reason": "Attendance + internal exam + assignment performance + final exam marks"
    },
    {
        "handle": "mahmoudelhemaly/students-grading-dataset",
        "reason": "Attendance + academic performance + student behavior signals"
    },
    {
        "handle": "pjaideep/synthetic-lpu-student-performance",
        "reason": "Attendance + internal marks + assignment scores + semester performance"
    },
    {
        "handle": "kundanbedmutha/student-performance-dataset",
        "reason": "Attendance and overall performance-related features"
    },
]

downloaded_path = None
selected_dataset = None

for item in PREFERRED_DATASETS:
    handle = item["handle"]
    try:
        print(f"Trying: {handle}")
        path = kagglehub.dataset_download(handle)
        downloaded_path = Path(path)
        selected_dataset = item
        print(f"Downloaded from: {handle}")
        print(f"Local path: {downloaded_path}")
        break
    except Exception as e:
        print(f"Failed for {handle}: {e}")

if downloaded_path is None:
    raise RuntimeError("No dataset could be downloaded. Check Kaggle authentication and rerun.")

In [ ]:

def list_dataset_files(root: Path):
    files = []
    for p in root.rglob("*"):
        if p.is_file():
            files.append(p)
    return sorted(files, key=lambda x: (x.suffix.lower(), x.name.lower()))

files = list_dataset_files(downloaded_path)
print(f"Selected dataset: {selected_dataset['handle']}")
print(f"Reason: {selected_dataset['reason']}")
print("\nFiles found:")
for f in files[:100]:
    print("-", f.name)

## 3. Load the most compatible dataset file

The notebook tries CSV / Excel / JSON files and chooses the dataframe that best matches:
- attendance-like columns
- score-like columns
- a plausible numeric target column such as final score / semester performance

In [ ]:

def clean_col_name(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

def try_read_table(path: Path):
    try:
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path)
        if path.suffix.lower() in [".xlsx", ".xls"]:
            return pd.read_excel(path)
        if path.suffix.lower() == ".json":
            return pd.read_json(path)
    except Exception:
        return None
    return None

TABULAR_EXTS = {".csv", ".xlsx", ".xls", ".json"}
candidate_tables = []

for f in files:
    if f.suffix.lower() in TABULAR_EXTS:
        df_try = try_read_table(f)
        if df_try is not None and not df_try.empty:
            df_try.columns = [clean_col_name(c) for c in df_try.columns]
            candidate_tables.append((f, df_try))

print(f"Loaded {len(candidate_tables)} candidate table(s).")
for f, df_try in candidate_tables[:20]:
    print(f"{f.name:45s} -> shape={df_try.shape}")

In [ ]:

TARGET_KEYWORDS = [
    "final_score", "final_exam_score", "final_exam_marks", "final_exam_mark",
    "final_marks", "exam_score", "semester_performance", "overall_score",
    "performance_score", "overall_performance", "final_result_score",
    "marks_obtained", "academic_score", "gpa", "performance_index"
]

ATTENDANCE_KEYWORDS = [
    "attendance_rate", "attendance_percentage", "attendance_percent",
    "attendance", "attendes_peresntage", "presence_rate"
]

ACADEMIC_FEATURE_HINTS = [
    "term", "internal", "assignment", "quiz", "test", "mid", "midterm",
    "project", "previous", "past", "score", "marks", "exam", "performance",
    "semester", "assessment"
]

def choose_target_column(df):
    cols = list(df.columns)
    numeric_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    for keyword in TARGET_KEYWORDS:
        hits = [c for c in numeric_cols if keyword in c]
        if hits:
            return hits[0]
    score_like = []
    for c in numeric_cols:
        series = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(series) == 0:
            continue
        in_range = ((series >= 0) & (series <= 100)).mean()
        if in_range >= 0.85 and any(h in c for h in ["score", "marks", "grade", "performance"]):
            score_like.append((c, len(series), in_range))
    if score_like:
        score_like = sorted(score_like, key=lambda x: (-x[2], -x[1]))
        return score_like[0][0]
    return None

def choose_attendance_column(df):
    cols = list(df.columns)
    for keyword in ATTENDANCE_KEYWORDS:
        hits = [c for c in cols if keyword in c]
        if hits:
            return hits[0]
    return None

def compatibility_score(df):
    score = 0
    target = choose_target_column(df)
    attendance = choose_attendance_column(df)
    if target:
        score += 5
    if attendance:
        score += 3
    for c in df.columns:
        if any(h in c for h in ACADEMIC_FEATURE_HINTS):
            score += 0.25
    numeric_cols = sum(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)
    score += min(numeric_cols, 20) * 0.1
    return score, target, attendance

table_scores = []
for f, df_try in candidate_tables:
    score, target_col, attendance_col = compatibility_score(df_try)
    table_scores.append({
        "file": f,
        "shape": df_try.shape,
        "score": score,
        "target_col": target_col,
        "attendance_col": attendance_col,
        "df": df_try
    })

table_scores = sorted(table_scores, key=lambda x: x["score"], reverse=True)
best = table_scores[0]
df_raw = best["df"].copy()

print("Selected table file:", best["file"].name)
print("Shape:", best["shape"])
print("Compatibility score:", best["score"])
print("Detected target column:", best["target_col"])
print("Detected attendance column:", best["attendance_col"])

display(df_raw.head())

## 4. Data cleaning

In [ ]:

df = df_raw.copy()

before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate row(s).")

for c in df.columns:
    if df[c].dtype == "object":
        numeric_version = pd.to_numeric(df[c], errors="coerce")
        if numeric_version.notna().mean() >= 0.75:
            df[c] = numeric_version

for c in df.columns:
    if df[c].dtype == "object":
        df[c] = df[c].astype(str).str.strip()

missing_report = (
    df.isna().sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values(["missing_pct", "missing_count"], ascending=False)
)
display(missing_report.head(20))

## 5. Build ScholarAI website-compatible features

Your Flask form expects:
- `term1_score`
- `term2_score`
- `term3_score`
- `attendance_rate`

Public Kaggle datasets may not use exactly the same names, so this notebook:
1. uses exact term-like columns if they exist, or
2. maps the strongest score-like numeric columns into the three term fields.

In [ ]:

target_col = choose_target_column(df)
attendance_col = choose_attendance_column(df)

if target_col is None:
    raise RuntimeError("Could not find a numeric target column for final score / performance.")

df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
if attendance_col:
    df[attendance_col] = pd.to_numeric(df[attendance_col], errors="coerce")

exact_term_candidates = {
    "term1_score": [c for c in df.columns if "term1" in c or "term_1" in c or "semester1" in c or "sem1" in c],
    "term2_score": [c for c in df.columns if "term2" in c or "term_2" in c or "semester2" in c or "sem2" in c],
    "term3_score": [c for c in df.columns if "term3" in c or "term_3" in c or "semester3" in c or "sem3" in c],
}

mapping_used = {}
website_df = pd.DataFrame(index=df.index)
website_df["attendance_rate"] = pd.to_numeric(df[attendance_col], errors="coerce") if attendance_col else np.nan
mapping_used["attendance_rate"] = attendance_col

used_cols = set([target_col, attendance_col])

if all(exact_term_candidates[k] for k in exact_term_candidates):
    website_df["term1_score"] = pd.to_numeric(df[exact_term_candidates["term1_score"][0]], errors="coerce")
    website_df["term2_score"] = pd.to_numeric(df[exact_term_candidates["term2_score"][0]], errors="coerce")
    website_df["term3_score"] = pd.to_numeric(df[exact_term_candidates["term3_score"][0]], errors="coerce")
    mapping_used["term1_score"] = exact_term_candidates["term1_score"][0]
    mapping_used["term2_score"] = exact_term_candidates["term2_score"][0]
    mapping_used["term3_score"] = exact_term_candidates["term3_score"][0]
else:
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    candidate_score_cols = []
    for c in numeric_cols:
        if c in used_cols:
            continue
        s = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(s) < max(30, int(0.1 * len(df))):
            continue
        in_range = ((s >= 0) & (s <= 100)).mean()
        score_name = any(k in c for k in ACADEMIC_FEATURE_HINTS)
        if in_range >= 0.70 and score_name:
            candidate_score_cols.append(c)

    ranked = []
    for c in candidate_score_cols:
        try:
            corr = abs(df[[c, target_col]].corr(numeric_only=True).iloc[0, 1])
            if pd.isna(corr):
                corr = 0
        except Exception:
            corr = 0
        ranked.append((c, corr))

    ranked = sorted(ranked, key=lambda x: x[1], reverse=True)
    top_cols = [c for c, _ in ranked[:3]]

    if len(top_cols) == 0:
        raise RuntimeError("Could not find usable score columns to map into term1/term2/term3.")
    elif len(top_cols) == 1:
        website_df["term1_score"] = df[top_cols[0]]
        website_df["term2_score"] = df[top_cols[0]]
        website_df["term3_score"] = df[top_cols[0]]
        mapping_used["term1_score"] = top_cols[0]
        mapping_used["term2_score"] = top_cols[0]
        mapping_used["term3_score"] = top_cols[0]
    elif len(top_cols) == 2:
        website_df["term1_score"] = df[top_cols[0]]
        website_df["term2_score"] = df[top_cols[1]]
        website_df["term3_score"] = (pd.to_numeric(df[top_cols[0]], errors="coerce") + pd.to_numeric(df[top_cols[1]], errors="coerce")) / 2
        mapping_used["term1_score"] = top_cols[0]
        mapping_used["term2_score"] = top_cols[1]
        mapping_used["term3_score"] = f"average({top_cols[0]}, {top_cols[1]})"
    else:
        website_df["term1_score"] = df[top_cols[0]]
        website_df["term2_score"] = df[top_cols[1]]
        website_df["term3_score"] = df[top_cols[2]]
        mapping_used["term1_score"] = top_cols[0]
        mapping_used["term2_score"] = top_cols[1]
        mapping_used["term3_score"] = top_cols[2]

website_df["target_score"] = pd.to_numeric(df[target_col], errors="coerce")
website_df["terminal_avg"] = website_df[["term1_score", "term2_score", "term3_score"]].mean(axis=1)

for c in ["attendance_rate", "term1_score", "term2_score", "term3_score", "terminal_avg", "target_score"]:
    website_df[c] = pd.to_numeric(website_df[c], errors="coerce")
    website_df[c] = website_df[c].clip(lower=0, upper=100)

print("Mapping used for ScholarAI features:")
print(json.dumps(mapping_used, indent=2))
display(website_df.head())

In [ ]:

website_df = website_df.dropna(subset=["target_score"]).reset_index(drop=True)
website_df = website_df[website_df[["attendance_rate", "term1_score", "term2_score", "term3_score"]].notna().sum(axis=1) >= 3].reset_index(drop=True)

print("Final modelling dataframe shape:", website_df.shape)
display(website_df.describe().T)

## 6. Exploratory analysis

In [ ]:

plt.figure(figsize=(8, 5))
website_df["target_score"].hist(bins=20)
plt.title("Target Score Distribution")
plt.xlabel("Target Score")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(website_df["attendance_rate"], website_df["target_score"])
plt.title("Attendance vs Target Score")
plt.xlabel("Attendance Rate")
plt.ylabel("Target Score")
plt.show()

corr = website_df[["attendance_rate", "term1_score", "term2_score", "term3_score", "terminal_avg", "target_score"]].corr(numeric_only=True)
display(corr)

## 7. Risk label function

This follows your Flask app thresholds:
- `low` if score >= 75
- `medium` if score >= 55 and < 75
- `high` if score < 55

In [ ]:

def score_to_risk(score):
    if score >= 75:
        return "low"
    elif score >= 55:
        return "medium"
    return "high"

website_df["actual_risk"] = website_df["target_score"].apply(score_to_risk)
website_df["actual_risk"].value_counts(dropna=False)

## 8. Train/test split and model comparison

In [ ]:

FEATURES = ["attendance_rate", "term1_score", "term2_score", "term3_score", "terminal_avg"]
TARGET = "target_score"

X = website_df[FEATURES].copy()
y = website_df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

def build_pipeline(model, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(steps=steps)
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, FEATURES)
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

models = {
    "LinearRegression": build_pipeline(LinearRegression(), scale=False),
    "Ridge": build_pipeline(Ridge(random_state=RANDOM_STATE), scale=True),
    "RandomForest": build_pipeline(
        RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        scale=False
    ),
    "ExtraTrees": build_pipeline(
        ExtraTreesRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
        scale=False
    ),
    "GradientBoosting": build_pipeline(
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        scale=False
    ),
}

In [ ]:

def evaluate_regression_and_risk(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = math.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    actual_risk = pd.Series(y_test).apply(score_to_risk)
    pred_risk = pd.Series(pred).apply(score_to_risk)

    acc = accuracy_score(actual_risk, pred_risk)
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual_risk, pred_risk, average="macro", zero_division=0
    )

    return {
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "Risk_Accuracy": round(acc, 4),
        "Risk_Precision_Macro": round(precision, 4),
        "Risk_Recall_Macro": round(recall, 4),
        "Risk_F1_Macro": round(f1, 4),
        "predictions": pred
    }

results = {}
for name, model in models.items():
    results[name] = evaluate_regression_and_risk(model, X_train, X_test, y_train, y_test)

results_df = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "predictions"} for k, v in results.items()}).T
results_df = results_df.sort_values(["MAE", "Risk_F1_Macro"], ascending=[True, False])
display(results_df)

## 9. Cross-validation on top candidates

In [ ]:

top_candidates = list(results_df.head(2).index)
cv_rows = []

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name in top_candidates:
    model = models[name]
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "model": name,
        "cv_mae_mean": round(-scores["test_mae"].mean(), 4),
        "cv_rmse_mean": round(-scores["test_rmse"].mean(), 4),
        "cv_r2_mean": round(scores["test_r2"].mean(), 4),
    })

cv_df = pd.DataFrame(cv_rows).sort_values(["cv_mae_mean", "cv_r2_mean"], ascending=[True, False])
display(cv_df)

## 10. Hyperparameter tuning

In [ ]:

param_spaces = {
    "RandomForest": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [None, 6, 10, 14],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "ExtraTrees": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [None, 8, 12, 16],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "GradientBoosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
        "model__subsample": [0.8, 1.0]
    },
    "Ridge": {
        "model__alpha": [0.1, 1.0, 10.0, 50.0]
    }
}

best_baseline_name = results_df.index[0]
print("Best baseline model:", best_baseline_name)

base_model = models[best_baseline_name]
param_grid = param_spaces.get(best_baseline_name, None)

if param_grid:
    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    print("Best params:", grid.best_params_)
else:
    print("No param grid defined; using baseline model as final.")
    best_model = base_model.fit(X_train, y_train)

In [ ]:

final_pred = best_model.predict(X_test)

final_metrics = {
    "MAE": round(mean_absolute_error(y_test, final_pred), 4),
    "RMSE": round(math.sqrt(mean_squared_error(y_test, final_pred)), 4),
    "R2": round(r2_score(y_test, final_pred), 4),
}

actual_risk = pd.Series(y_test).apply(score_to_risk)
pred_risk = pd.Series(final_pred).apply(score_to_risk)

final_metrics["Risk_Accuracy"] = round(accuracy_score(actual_risk, pred_risk), 4)
precision, recall, f1, _ = precision_recall_fscore_support(
    actual_risk, pred_risk, average="macro", zero_division=0
)
final_metrics["Risk_Precision_Macro"] = round(precision, 4)
final_metrics["Risk_Recall_Macro"] = round(recall, 4)
final_metrics["Risk_F1_Macro"] = round(f1, 4)

print("Final tuned model metrics:")
display(pd.DataFrame([final_metrics]))
print("\nRisk classification report:")
print(classification_report(actual_risk, pred_risk, zero_division=0))

In [ ]:

plt.figure(figsize=(7, 5))
plt.scatter(y_test, final_pred)
plt.xlabel("Actual Target Score")
plt.ylabel("Predicted Score")
plt.title("Actual vs Predicted Score")
plt.show()

cm = confusion_matrix(actual_risk, pred_risk, labels=["high", "medium", "low"])
cm_df = pd.DataFrame(cm, index=["actual_high", "actual_medium", "actual_low"], columns=["pred_high", "pred_medium", "pred_low"])
display(cm_df)

## 11. Feature importance / model interpretation

In [ ]:

feature_names = FEATURES
model_obj = best_model.named_steps["model"]

importance_df = None
if hasattr(model_obj, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model_obj.feature_importances_
    }).sort_values("importance", ascending=False)
elif hasattr(model_obj, "coef_"):
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": np.abs(model_obj.coef_)
    }).sort_values("importance", ascending=False)

if importance_df is not None:
    display(importance_df)
    plt.figure(figsize=(8, 5))
    plt.bar(importance_df["feature"], importance_df["importance"])
    plt.title("Feature Importance")
    plt.xlabel("Feature")
    plt.ylabel("Importance")
    plt.show()
else:
    print("This model does not expose direct feature importance.")

## 12. Business recommendation engine

This recommendation function is designed for the web app output.

It keeps the model focused on **predicted score**, while the website can still combine that score later with:
- dues
- complaint logs
- trend
- other live operational signals

In [ ]:

def build_business_recommendation(predicted_score, attendance_rate, term1_score, term2_score, term3_score):
    recommendations = []

    if predicted_score < 55:
        risk_level = "high"
    elif predicted_score < 75:
        risk_level = "medium"
    else:
        risk_level = "low"

    if attendance_rate < 65:
        recommendations.append("Critical attendance: arrange immediate follow-up with guardian and class teacher.")
    elif attendance_rate < 75:
        recommendations.append("Low attendance: encourage consistent attendance and monitor weekly.")

    if predicted_score < 55:
        recommendations.append("Provide urgent academic intervention, remedial sessions, and progress tracking.")
    elif predicted_score < 75:
        recommendations.append("Provide targeted support in weak areas and schedule additional practice.")
    else:
        recommendations.append("Maintain current progress and continue regular monitoring.")

    score_span = max(term1_score, term2_score, term3_score) - min(term1_score, term2_score, term3_score)
    if score_span >= 15:
        recommendations.append("Performance is unstable across terms; review subject-level gaps and study plan.")

    trend_1 = term2_score - term1_score
    trend_2 = term3_score - term2_score
    if trend_1 < -3 and trend_2 < -3:
        recommendations.append("Declining trend detected; plan early intervention before final assessment.")

    return risk_level, " ".join(recommendations)

sample_recs = website_df.head(10).copy()
sample_recs["sample_predicted_score"] = best_model.predict(sample_recs[FEATURES]).round(2)
sample_recs[["risk_level", "business_recommendation"]] = sample_recs.apply(
    lambda row: pd.Series(
        build_business_recommendation(
            predicted_score=float(row["sample_predicted_score"]),
            attendance_rate=float(row["attendance_rate"]),
            term1_score=float(row["term1_score"]),
            term2_score=float(row["term2_score"]),
            term3_score=float(row["term3_score"]),
        )
    ),
    axis=1
)
display(sample_recs[FEATURES + ["sample_predicted_score", "risk_level", "business_recommendation"]])

## 13. Export artifacts for Flask

These files are the main deliverables from the notebook:

- **`scholarai_academic_score_pipeline.joblib`** → load in Flask for live prediction
- **`scholarai_model_metadata.json`** → feature names, dataset mapping, metrics
- **`scholarai_business_rules.json`** → risk thresholds and recommendations reference

In [ ]:

artifact_dir = Path("/content/scholarai_artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "scholarai_academic_score_pipeline.joblib"
meta_path = artifact_dir / "scholarai_model_metadata.json"
rules_path = artifact_dir / "scholarai_business_rules.json"

joblib.dump(best_model, model_path)

metadata = {
    "project": "ScholarAI",
    "purpose": "Predict academic score for Flask predicted_score column",
    "selected_dataset_handle": selected_dataset["handle"],
    "selected_dataset_reason": selected_dataset["reason"],
    "selected_table_file": str(best["file"].name),
    "source_to_website_feature_mapping": mapping_used,
    "website_features": FEATURES,
    "target_column_in_training_data": target_col,
    "final_metrics": final_metrics,
    "risk_thresholds": {
        "low_min_score": 75,
        "medium_min_score": 55,
        "high_below_score": 55
    },
    "notes": [
        "Model predicts academic score only.",
        "Flask should calculate final risk using live dues, complaint logs, attendance and trend.",
        "Use the same shared service in both admin and student routes."
    ]
}

business_rules = {
    "risk_thresholds": {
        "low_min_score": 75,
        "medium_min_score": 55,
        "high_below_score": 55
    },
    "recommendation_rules": {
        "attendance_lt_65": "Critical attendance: arrange immediate follow-up with guardian and class teacher.",
        "attendance_lt_75": "Low attendance: encourage consistent attendance and monitor weekly.",
        "predicted_score_lt_55": "Provide urgent academic intervention, remedial sessions, and progress tracking.",
        "predicted_score_lt_75": "Provide targeted support in weak areas and schedule additional practice.",
        "performance_unstable": "Performance is unstable across terms; review subject-level gaps and study plan.",
        "declining_trend": "Declining trend detected; plan early intervention before final assessment."
    }
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

with open(rules_path, "w", encoding="utf-8") as f:
    json.dump(business_rules, f, indent=2)

print("Saved:")
print(model_path)
print(meta_path)
print(rules_path)

## 14. Flask-compatible prediction function

In [ ]:

loaded_model = joblib.load(model_path)

def predict_score_for_flask(attendance_rate, term1_score, term2_score, term3_score):
    terminal_avg = (term1_score + term2_score + term3_score) / 3

    input_df = pd.DataFrame([{
        "attendance_rate": attendance_rate,
        "term1_score": term1_score,
        "term2_score": term2_score,
        "term3_score": term3_score,
        "terminal_avg": terminal_avg
    }])

    predicted_score = float(loaded_model.predict(input_df)[0])
    predicted_score = round(max(0, min(100, predicted_score)), 2)

    risk_level, recommendation = build_business_recommendation(
        predicted_score=predicted_score,
        attendance_rate=attendance_rate,
        term1_score=term1_score,
        term2_score=term2_score,
        term3_score=term3_score
    )

    return {
        "predicted_score": predicted_score,
        "risk_level": risk_level,
        "recommendation": recommendation
    }

demo = predict_score_for_flask(
    attendance_rate=78,
    term1_score=64,
    term2_score=58,
    term3_score=62
)
demo

## 15. Download the exported files

In Colab, run this if you want to download the artifacts:

```python
from google.colab import files
files.download('/content/scholarai_artifacts/scholarai_academic_score_pipeline.joblib')
files.download('/content/scholarai_artifacts/scholarai_model_metadata.json')
files.download('/content/scholarai_artifacts/scholarai_business_rules.json')
```